In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

### Load data

In [ ]:
df_New=pd.read_csv('Humandataset.csv')
df_New.head()

In [ ]:
df_New.drop(columns='Unnamed: 0',inplace=True)


In [ ]:
df_New.iloc[0,:30]

In [ ]:
df_New.isnull().sum().all()

In [ ]:
df_New.duplicated().sum().all()

In [ ]:
df_New.describe()

In [ ]:
plt.figure(figsize = (12,8))
sns.countplot(x = 'Activity', data = df_New);

In [ ]:
X=df_New.iloc[:,:-2]
y=df_New.iloc[:,-1:]

# Classification Phase


### Splitting data to train and test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

During train uncomment this cell

In [ ]:
print('X_train_shape ---->',X_train.shape)
print('X_test_shape ---->',X_test.shape)
print('Y_train_shape ---->',y_train.shape)
print('Y_test_shape ---->',y_test.shape)

During train uncomment this cell

### Preprocessimg label data and feature

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_labeled=le.fit_transform(y_train)
y_labeled_test=le.transform(y_test)
Y_all=le.transform(y)


In [ ]:
z=le.inverse_transform(np.unique(y_labeled))


In [ ]:
z

In [ ]:
from sklearn.preprocessing import StandardScaler

sc=StandardScaler()
x_Scaler_train=sc.fit_transform(X_train)
x_Scaler_test=sc.transform(X_test)

X_all=sc.transform(X)


### Using Support vector machine Classification

In [ ]:
from sklearn.svm import SVC
classifier_SVC=SVC(kernel = 'linear',C=1,gamma=0.001)
x=classifier_SVC.fit(x_Scaler_train,y_labeled)





In [ ]:
cv_svm_train = cross_val_score(classifier_SVC , X_all , Y_all, cv = 10)
print("Cross validation for train data SVC is : ", cv_svm_train)

During train uncomment this cell

In [ ]:
y_pred_SVC=classifier_SVC.predict(x_Scaler_test)
cm = confusion_matrix(y_labeled_test,y_pred_SVC)
print("Confusion matrix is :\n ", cm)

annot_kws = {"ha": 'left',"va": 'center','size': 9}
plt.figure(figsize=(10,6))
ax= plt.subplot()
sns.heatmap(cm, annot=True, fmt='g',cmap = 'Blues',annot_kws=annot_kws, ax=ax); 
bottom, top = ax.get_ylim()
ax.set_ylim(bottom + 0.5, top - 0.5)
# labels, title and ticks
ax.set_xlabel('Predicted labels',fontsize=16,color='c')
ax.set_ylabel('True labels',fontsize=16,color='c'); 
ax.set_title('SVM Confusion Matrix',fontsize=16,color='c'); 
plt.xticks(fontsize=10)
plt.yticks(fontsize=10,rotation=0)
ax.xaxis.set_ticklabels(z,rotation=10);
ax.yaxis.set_ticklabels(z);
plt.savefig('ss.jpg')


### Showing Accuracy using accuracy_score

In [ ]:
from sklearn.metrics import  accuracy_score

print('Accuracy of SVC--------->',round(accuracy_score(y_labeled_test, y_pred_SVC),2))


In [ ]:
x=list(set(y_labeled_test))
x


In [ ]:
from sklearn.metrics import RocCurveDisplay
for i in list(set(y_labeled_test)):
    RocCurveDisplay.from_predictions(y_labeled_test, y_pred_SVC,pos_label=i)
    plt.title('ROC for '+ z[i])
    plt.legend(loc='upper center')
    plt.savefig('ROC for '+ z[i]+'.jpg')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.model_selection import learning_curve
from sklearn.model_selection import ShuffleSplit


def plot_learning_curve(
    estimator,
    title,
    X,
    y,
    axes=None,
    ylim=None,
    cv=None,
    n_jobs=None,
    train_sizes=np.linspace(0.1, 1.0, 5),
):
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(20, 5))

        axes[0].set_title(title)
    if ylim is not None:
        axes[0].set_ylim(*ylim)
        axes[0].set_xlabel("Training examples")
        axes[0].set_ylabel("Score")

    train_sizes, train_scores, test_scores, fit_times, _ = learning_curve(
        estimator,
        X,
        y,
        cv=cv,
        n_jobs=n_jobs,
        train_sizes=train_sizes,
        return_times=True,
    )
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    fit_times_mean = np.mean(fit_times, axis=1)
    fit_times_std = np.std(fit_times, axis=1)

    # Plot learning curve
    axes[0].grid()
    axes[0].fill_between(
        train_sizes,
        train_scores_mean - train_scores_std,
        train_scores_mean + train_scores_std,
        alpha=0.1,
        color="r",
    )
    axes[0].fill_between(
        train_sizes,
        test_scores_mean - test_scores_std,
        test_scores_mean + test_scores_std,
        alpha=0.1,
        color="g",
    )
    axes[0].plot(
        train_sizes, train_scores_mean, "o-", color="r", label="Training score"
    )
    axes[0].plot(
        train_sizes, test_scores_mean, "o-", color="g", label="Cross-validation score"
    )
    axes[0].legend(loc="best")

    # Plot n_samples vs fit_times
    axes[1].grid()
    axes[1].plot(train_sizes, fit_times_mean, "o-")
    axes[1].fill_between(
        train_sizes,
        fit_times_mean - fit_times_std,
        fit_times_mean + fit_times_std,
        alpha=0.1,
    )
    axes[1].set_xlabel("Training examples")
    axes[1].set_ylabel("fit_times")
    axes[1].set_title("Scalability of the model")

    # Plot fit_time vs score
    fit_time_argsort = fit_times_mean.argsort()
    fit_time_sorted = fit_times_mean[fit_time_argsort]
    test_scores_mean_sorted = test_scores_mean[fit_time_argsort]
    test_scores_std_sorted = test_scores_std[fit_time_argsort]
    axes[2].grid()
    axes[2].plot(fit_time_sorted, test_scores_mean_sorted, "o-")
    axes[2].fill_between(
        fit_time_sorted,
        test_scores_mean_sorted - test_scores_std_sorted,
        test_scores_mean_sorted + test_scores_std_sorted,
        alpha=0.1,
    )
    axes[2].set_xlabel("fit_times")
    axes[2].set_ylabel("Score")
    axes[2].set_title("Performance of the model")

    return plt


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 15))
title = r"Learning Curves (SVM, Linear kernel, $\gamma=0.001$)"
# SVC is more expensive so we do a lower number of CV iterations:
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)
estimator = SVC(kernel = 'linear',C=1,gamma=0.001)
plot_learning_curve(
    estimator, title, X, y, axes=axes[:], ylim=(0.7, 1.01), cv=cv, n_jobs=4
)
plt.show()

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.linear_model import Ridge
train_scores, valid_scores = validation_curve(
   Ridge(), X, y, param_name="alpha", param_range=np.logspace(-7, 3, 3),
   cv=5)

In [ ]:
import sys
from io import StringIO
old_stdout = sys.stdout
sys.stdout = mystdout = StringIO()
classifier_SVC=SVC(kernel = 'linear',C=1,gamma=0.001)
classifier_SVC.fit(x_Scaler_train,y_labeled)
sys.stdout = old_stdout
loss_history = mystdout.getvalue()
loss_list = []
for line in loss_history.split('\n'):
    if(len(line.split("loss: ")) == 1):
        continue
    loss_list.append(float(line.split("loss: ")[-1]))

In [ ]:
plt.figure()
plt.plot(np.arange(len(loss_list)), loss_list)
plt.xlabel("Time in epochs")
plt.ylabel("Loss")
plt.close()

In [ ]:
loss_history